# Meta-Learning — Aprender a Aprender

## Notebook educativo — Implementación desde cero

### Objetivo
Comprender y construir desde cero los principales paradigmas de **Meta-Learning**: sistemas que aprenden de la experiencia en múltiples tareas para adaptarse rápidamente a tareas nuevas con pocos datos.

### Contenido
1. **¿Qué es Meta-Learning?** — Motivación, taxonomía y diferencias con ML tradicional
2. **Learning to Learn: el marco formal** — Meta-train, meta-test, episodios, soporte y query
3. **Metric-Based Meta-Learning** — Siamese Networks, Matching Networks, Prototypical Networks
4. **Prototypical Networks** — Implementación completa desde cero
5. **Model-Based Meta-Learning** — MANN (Memory-Augmented Neural Networks)
6. **Optimization-Based Meta-Learning** — MAML (Model-Agnostic Meta-Learning)
7. **MAML desde cero** — Implementación completa con inner/outer loop
8. **Reptile** — Versión simplificada de MAML
9. **Meta-Learning para selección de algoritmos** — Recomendación basada en meta-features
10. **Comparación y experimentos** — Few-shot, convergencia, ablation study

### Requisitos
```
numpy matplotlib scikit-learn
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, load_digits, load_iris
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from copy import deepcopy
import warnings
import time
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
np.random.seed(42)
print("✅ Entorno listo")

## 1. ¿Qué es Meta-Learning?

### El problema del ML tradicional
El ML tradicional necesita **muchos datos** para cada tarea nueva:

```
Tarea 1: 10,000 imágenes de gatos/perros    → Modelo 1
Tarea 2: 10,000 imágenes de coches/aviones   → Modelo 2 (desde cero)
Tarea 3: 10,000 imágenes de flores/árboles   → Modelo 3 (desde cero)
```

Un humano, en cambio, puede aprender un concepto nuevo con **1-2 ejemplos**.

### La idea de Meta-Learning
En vez de aprender **una tarea**, aprender un **proceso de aprendizaje** que funcione para muchas tareas:

```
Meta-entrenamiento (muchas tareas):
  Tarea 1: gatos/perros        → extraer conocimiento
  Tarea 2: coches/aviones      → extraer conocimiento    → Meta-conocimiento
  Tarea 3: flores/árboles      → extraer conocimiento
  ...
  Tarea 100: sillas/mesas      → extraer conocimiento

Meta-test (tarea nueva con pocos datos):
  Tarea nueva: delfines/ballenas (solo 5 ejemplos) → Meta-conocimiento → ¡Buen modelo!
```

### Definición formal
$$\theta^* = \arg\min_\theta \sum_{\mathcal{T}_i \sim p(\mathcal{T})} \mathcal{L}(\theta, \mathcal{T}_i)$$

Donde:
- $p(\mathcal{T})$: distribución de tareas
- $\mathcal{T}_i = (D_i^{\text{support}}, D_i^{\text{query}})$: tarea con sus datos
- $\theta$: meta-parámetros (lo que se comparte entre tareas)
- $\mathcal{L}$: pérdida evaluada en el query set después de adaptar con el support set

### Taxonomía

| Paradigma | Idea | Ejemplos |
|-----------|------|----------|
| **Metric-based** | Aprender un espacio de distancias | Siamese, Prototypical, Matching Networks |
| **Model-based** | Red que lee ejemplos y predice | MANN, SNAIL, MetaNet |
| **Optimization-based** | Aprender inicialización/optimizador | MAML, Reptile, Meta-SGD |

## 2. El Marco de Few-Shot Learning

### Terminología clave

**N-way K-shot**: clasificar entre $N$ clases usando solo $K$ ejemplos por clase.

- **5-way 1-shot**: 5 clases, 1 ejemplo cada una → 5 ejemplos totales
- **5-way 5-shot**: 5 clases, 5 ejemplos cada una → 25 ejemplos totales
- **Support set** $S$: los $K$ ejemplos por clase (datos de "entrenamiento" de la tarea)
- **Query set** $Q$: ejemplos para evaluar (datos de "test" de la tarea)

### Estructura de un episodio

```
Episodio i:
  ┌─ Support Set (K ejemplos por clase) ─────────────────┐
  │  Clase A: [img1_A, img2_A, ..., imgK_A]              │
  │  Clase B: [img1_B, img2_B, ..., imgK_B]              │
  │  ...                                                  │
  │  Clase N: [img1_N, img2_N, ..., imgK_N]              │
  └───────────────────────────────────────────────────────┘
                          ↓ adaptar
  ┌─ Query Set (evaluar) ────────────────────────────────┐
  │  [query1, query2, ..., queryM]  →  predecir clases   │
  └───────────────────────────────────────────────────────┘
```

### Meta-train vs Meta-test
- **Meta-train**: muchos episodios con clases del **training set** → aprender el meta-conocimiento
- **Meta-test**: episodios con clases **nunca vistas** → evaluar generalización

In [ ]:
# ============================================================
# Generador de Tareas Few-Shot
# ============================================================

class GeneradorTareasFewShot:
    """Genera episodios N-way K-shot para meta-learning."""
    
    def __init__(self, X, y, n_way=5, k_shot=5, q_queries=15):
        self.X = X
        self.y = y
        self.n_way = n_way
        self.k_shot = k_shot
        self.q_queries = q_queries
        self.clases = np.unique(y)
    
    def generar_episodio(self, clases_disponibles=None):
        """Genera un episodio: support set + query set."""
        if clases_disponibles is None:
            clases_disponibles = self.clases
        
        # Seleccionar N clases aleatorias
        clases_episodio = np.random.choice(clases_disponibles, 
                                            size=self.n_way, replace=False)
        
        support_X, support_y = [], []
        query_X, query_y = [], []
        
        for nueva_label, clase in enumerate(clases_episodio):
            # Índices de esta clase
            idx_clase = np.where(self.y == clase)[0]
            idx_seleccionados = np.random.choice(idx_clase, 
                                                  size=self.k_shot + self.q_queries,
                                                  replace=len(idx_clase) < self.k_shot + self.q_queries)
            
            # Dividir en support y query
            support_X.append(self.X[idx_seleccionados[:self.k_shot]])
            support_y.extend([nueva_label] * self.k_shot)
            
            query_X.append(self.X[idx_seleccionados[self.k_shot:self.k_shot + self.q_queries]])
            query_y.extend([nueva_label] * self.q_queries)
        
        support_X = np.vstack(support_X)
        support_y = np.array(support_y)
        query_X = np.vstack(query_X)
        query_y = np.array(query_y)
        
        return support_X, support_y, query_X, query_y, clases_episodio
    
    def generar_lote(self, n_episodios, clases_disponibles=None):
        """Genera un lote de episodios."""
        episodios = []
        for _ in range(n_episodios):
            episodios.append(self.generar_episodio(clases_disponibles))
        return episodios

# --- Crear dataset y demostrar ---
digits = load_digits()
X_digits = digits.data / 16.0  # Normalizar [0, 1]
y_digits = digits.target

# Split en meta-train (clases 0-6) y meta-test (clases 7-9)
clases_train = np.array([0, 1, 2, 3, 4, 5, 6])
clases_test = np.array([7, 8, 9])

print("=== Dataset: Dígitos escritos a mano (8x8) ===")
print(f"Total: {len(X_digits)} ejemplos, {len(np.unique(y_digits))} clases")
print(f"Clases meta-train: {clases_train} (para aprender)")
print(f"Clases meta-test:  {clases_test} (nunca vistas durante entrenamiento)")

# Generar un episodio de ejemplo
gen = GeneradorTareasFewShot(X_digits, y_digits, n_way=3, k_shot=5, q_queries=10)
s_X, s_y, q_X, q_y, clases = gen.generar_episodio(clases_train)

print(f"\n--- Episodio de ejemplo (3-way 5-shot) ---")
print(f"Clases seleccionadas: dígitos {clases}")
print(f"Support set: {s_X.shape} ({len(s_y)} ejemplos)")
print(f"Query set:   {q_X.shape} ({len(q_y)} ejemplos)")

# Visualizar
fig, axes = plt.subplots(3, 7, figsize=(14, 6))
for i in range(3):  # 3 clases
    mask = s_y == i
    ejemplos = s_X[mask][:5]
    for j in range(min(5, len(ejemplos))):
        axes[i, j].imshow(ejemplos[j].reshape(8, 8), cmap='gray')
        axes[i, j].axis('off')
        if j == 0:
            axes[i, j].set_title(f'Clase {i}\n(dígito {clases[i]})', fontweight='bold')
    
    # Query examples
    mask_q = q_y == i
    q_ej = q_X[mask_q][:2]
    for j in range(min(2, len(q_ej))):
        axes[i, 5+j].imshow(q_ej[j].reshape(8, 8), cmap='Reds')
        axes[i, 5+j].axis('off')
        if j == 0:
            axes[i, 5+j].set_title('Query', color='red', fontweight='bold')

plt.suptitle('Episodio Few-Shot: Support (gris) y Query (rojo)', fontweight='bold', fontsize=13)
plt.tight_layout(); plt.show()

## 3. Metric-Based Meta-Learning

### Idea central
Aprender una **función de distancia** (o un espacio de embeddings) donde ejemplos de la misma clase estén cerca y de clases diferentes estén lejos.

### Siamese Networks (Koch et al., 2015)
Dos entradas pasan por la **misma red** y se compara la distancia:

$$p(\text{misma clase} | x_1, x_2) = \sigma(\|f_\theta(x_1) - f_\theta(x_2)\|)$$

### Matching Networks (Vinyals et al., 2016)
Clasificar un query $\hat{x}$ como promedio ponderado por atención sobre el support set:

$$\hat{y} = \sum_{i=1}^{k} a(\hat{x}, x_i) \cdot y_i \quad \text{donde} \quad a(\hat{x}, x_i) = \text{softmax}(\cos(f(\hat{x}), g(x_i)))$$

### Prototypical Networks (Snell et al., 2017) ⭐
La idea más elegante: cada clase se representa por su **prototipo** (centroide en el espacio de embeddings):

$$c_k = \frac{1}{|S_k|} \sum_{(x_i, y_i) \in S_k} f_\theta(x_i)$$

Y se clasifica por **distancia al prototipo más cercano**:

$$p(y=k | x) = \frac{\exp(-d(f_\theta(x), c_k))}{\sum_{k'} \exp(-d(f_\theta(x), c_{k'}))}$$

donde $d$ es la distancia euclidiana al cuadrado.

In [ ]:
# ============================================================
# Prototypical Networks desde cero
# ============================================================

class RedEmbedding:
    """Red neuronal simple para generar embeddings.
    Arquitectura: input → hidden1 → hidden2 → embedding_dim
    """
    
    def __init__(self, input_dim, hidden_dim=64, embedding_dim=32):
        self.embedding_dim = embedding_dim
        # Inicialización He
        self.W1 = np.random.randn(input_dim, hidden_dim) * np.sqrt(2.0 / input_dim)
        self.b1 = np.zeros(hidden_dim)
        self.W2 = np.random.randn(hidden_dim, hidden_dim) * np.sqrt(2.0 / hidden_dim)
        self.b2 = np.zeros(hidden_dim)
        self.W3 = np.random.randn(hidden_dim, embedding_dim) * np.sqrt(2.0 / hidden_dim)
        self.b3 = np.zeros(embedding_dim)
    
    def forward(self, X):
        """Forward pass: X → embedding."""
        self.z1 = X @ self.W1 + self.b1
        self.h1 = np.maximum(0, self.z1)  # ReLU
        self.z2 = self.h1 @ self.W2 + self.b2
        self.h2 = np.maximum(0, self.z2)  # ReLU
        self.z3 = self.h2 @ self.W3 + self.b3
        return self.z3  # Embedding (sin activación)
    
    def get_params(self):
        return [self.W1, self.b1, self.W2, self.b2, self.W3, self.b3]
    
    def set_params(self, params):
        self.W1, self.b1, self.W2, self.b2, self.W3, self.b3 = params


class PrototypicalNetwork:
    """Prototypical Networks (Snell et al., 2017) desde cero."""
    
    def __init__(self, input_dim, embedding_dim=32, lr=0.001):
        self.encoder = RedEmbedding(input_dim, hidden_dim=64, embedding_dim=embedding_dim)
        self.lr = lr
        self.losses = []
    
    def _distancia_euclidiana(self, a, b):
        """Distancia euclidiana al cuadrado entre a (N, D) y b (M, D)."""
        # a: (N, D), b: (M, D) → resultado: (N, M)
        return np.sum((a[:, np.newaxis, :] - b[np.newaxis, :, :]) ** 2, axis=2)
    
    def _calcular_prototipos(self, support_X, support_y, n_way):
        """Calcula el centroide (prototipo) de cada clase."""
        embeddings = self.encoder.forward(support_X)
        prototipos = np.zeros((n_way, embeddings.shape[1]))
        for k in range(n_way):
            mask = support_y == k
            prototipos[k] = embeddings[mask].mean(axis=0)
        return prototipos
    
    def _forward_episodio(self, support_X, support_y, query_X, query_y, n_way):
        """Forward pass completo de un episodio."""
        # 1. Obtener prototipos del support set
        prototipos = self._calcular_prototipos(support_X, support_y, n_way)
        
        # 2. Embeddings del query set
        query_embeddings = self.encoder.forward(query_X)
        
        # 3. Distancias query → prototipos
        dists = self._distancia_euclidiana(query_embeddings, prototipos)  # (Q, N)
        
        # 4. Softmax sobre -distancias → probabilidades
        logits = -dists
        logits -= logits.max(axis=1, keepdims=True)  # Estabilidad
        exp_logits = np.exp(logits)
        probs = exp_logits / exp_logits.sum(axis=1, keepdims=True)
        
        # 5. Loss: cross-entropy
        n_query = len(query_y)
        loss = -np.sum(np.log(probs[np.arange(n_query), query_y] + 1e-10)) / n_query
        
        # 6. Accuracy
        preds = np.argmax(probs, axis=1)
        accuracy = np.mean(preds == query_y)
        
        return loss, accuracy, probs, prototipos, query_embeddings
    
    def _backward_episodio(self, support_X, support_y, query_X, query_y, 
                           n_way, probs, prototipos, query_embeddings):
        """Backward pass simplificado con diferencias finitas (numerical gradient)."""
        params = self.encoder.get_params()
        grads = []
        eps = 1e-4
        
        for p_idx, param in enumerate(params):
            grad = np.zeros_like(param)
            # Muestrear un subset de parámetros para eficiencia
            n_sample = min(param.size, 200)
            indices = np.random.choice(param.size, n_sample, replace=False)
            
            flat_param = param.ravel()
            for idx in indices:
                orig = flat_param[idx]
                
                flat_param[idx] = orig + eps
                loss_plus, _, _, _, _ = self._forward_episodio(
                    support_X, support_y, query_X, query_y, n_way)
                
                flat_param[idx] = orig - eps
                loss_minus, _, _, _, _ = self._forward_episodio(
                    support_X, support_y, query_X, query_y, n_way)
                
                flat_param[idx] = orig
                grad.ravel()[idx] = (loss_plus - loss_minus) / (2 * eps)
            
            grads.append(grad)
        
        # Actualizar parámetros
        for i, (param, grad) in enumerate(zip(params, grads)):
            params[i] = param - self.lr * grad
        
        self.encoder.set_params(params)
    
    def entrenar(self, generador, n_episodios=200, clases_train=None, verbose=True):
        """Meta-entrenamiento sobre múltiples episodios."""
        if verbose:
            print("=== Entrenamiento Prototypical Network ===\n")
        
        for ep in range(n_episodios):
            s_X, s_y, q_X, q_y, clases = generador.generar_episodio(clases_train)
            n_way = len(np.unique(s_y))
            
            loss, acc, probs, protos, q_emb = self._forward_episodio(
                s_X, s_y, q_X, q_y, n_way)
            
            self.losses.append(loss)
            
            # Backward cada episodio
            self._backward_episodio(s_X, s_y, q_X, q_y, n_way, 
                                    probs, protos, q_emb)
            
            if verbose and (ep + 1) % 50 == 0:
                avg_loss = np.mean(self.losses[-50:])
                print(f"  Episodio {ep+1:4d}: loss={avg_loss:.4f}, acc={acc:.4f}")
        
        if verbose:
            print(f"\n→ Entrenamiento completado ({n_episodios} episodios)")
    
    def evaluar(self, generador, n_episodios=100, clases_test=None):
        """Evalúa en episodios con clases nunca vistas."""
        accuracies = []
        for _ in range(n_episodios):
            s_X, s_y, q_X, q_y, clases = generador.generar_episodio(clases_test)
            n_way = len(np.unique(s_y))
            _, acc, _, _, _ = self._forward_episodio(s_X, s_y, q_X, q_y, n_way)
            accuracies.append(acc)
        return np.mean(accuracies), np.std(accuracies)
    
    def predecir(self, support_X, support_y, query_X):
        """Predice clases para nuevas queries dado un support set."""
        n_way = len(np.unique(support_y))
        prototipos = self._calcular_prototipos(support_X, support_y, n_way)
        query_emb = self.encoder.forward(query_X)
        dists = self._distancia_euclidiana(query_emb, prototipos)
        return np.argmin(dists, axis=1)

# --- Entrenar Prototypical Network ---
gen_train = GeneradorTareasFewShot(X_digits, y_digits, n_way=3, k_shot=5, q_queries=10)
proto_net = PrototypicalNetwork(input_dim=64, embedding_dim=16, lr=0.005)
proto_net.entrenar(gen_train, n_episodios=200, clases_train=clases_train)

# Evaluar en clases nunca vistas
gen_test = GeneradorTareasFewShot(X_digits, y_digits, n_way=3, k_shot=5, q_queries=10)
acc_test, std_test = proto_net.evaluar(gen_test, n_episodios=50, clases_test=clases_test)
print(f"\n🎯 Accuracy en clases NUEVAS (7,8,9): {acc_test:.4f} ± {std_test:.4f}")
print(f"   (Random baseline para 3-way: {1/3:.4f})")

In [ ]:
# ============================================================
# Visualización: Espacio de embeddings y prototipos
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Curva de entrenamiento
ax = axes[0]
window = 20
losses_smooth = np.convolve(proto_net.losses, np.ones(window)/window, mode='valid')
ax.plot(losses_smooth, 'b-', lw=1.5)
ax.set_xlabel('Episodio'); ax.set_ylabel('Loss')
ax.set_title('Curva de entrenamiento\n(Prototypical Network)', fontweight='bold')
ax.grid(True, alpha=0.3)

# 2. Espacio de embeddings del support set (reducido a 2D con PCA simple)
ax = axes[1]
s_X, s_y, q_X, q_y, clases_ep = gen_test.generar_episodio(clases_test)
n_way = len(np.unique(s_y))

# Obtener embeddings
all_X = np.vstack([s_X, q_X])
all_emb = proto_net.encoder.forward(all_X)

# PCA manual a 2D
mean_emb = all_emb.mean(axis=0)
all_emb_c = all_emb - mean_emb
cov = all_emb_c.T @ all_emb_c / len(all_emb_c)
eigenvals, eigenvecs = np.linalg.eigh(cov)
idx = eigenvals.argsort()[::-1][:2]
proj = eigenvecs[:, idx]
emb_2d = all_emb_c @ proj

s_emb_2d = emb_2d[:len(s_X)]
q_emb_2d = emb_2d[len(s_X):]

# Prototipos
prototipos = proto_net._calcular_prototipos(s_X, s_y, n_way)
proto_c = prototipos - mean_emb
proto_2d = proto_c @ proj

colores = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6', '#f39c12']
for k in range(n_way):
    mask_s = s_y == k
    mask_q = q_y == k
    ax.scatter(s_emb_2d[mask_s, 0], s_emb_2d[mask_s, 1], 
              c=colores[k], marker='o', s=60, alpha=0.7, label=f'Support clase {k}')
    ax.scatter(q_emb_2d[mask_q, 0], q_emb_2d[mask_q, 1], 
              c=colores[k], marker='x', s=80, alpha=0.7)
    ax.scatter(proto_2d[k, 0], proto_2d[k, 1], 
              c=colores[k], marker='*', s=300, edgecolors='black', linewidths=1.5,
              zorder=5)

ax.set_title(f'Espacio de embeddings\n(★=prototipo, o=support, x=query)', fontweight='bold')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# 3. Accuracy vs K-shot
ax = axes[2]
k_shots = [1, 2, 3, 5, 10, 15]
accs_by_k = []

for k in k_shots:
    gen_k = GeneradorTareasFewShot(X_digits, y_digits, n_way=3, k_shot=k, q_queries=10)
    acc, std = proto_net.evaluar(gen_k, n_episodios=30, clases_test=clases_test)
    accs_by_k.append((acc, std))

accs = [a[0] for a in accs_by_k]
stds = [a[1] for a in accs_by_k]
ax.errorbar(k_shots, accs, yerr=stds, fmt='bo-', lw=2, capsize=5, markersize=8)
ax.axhline(1/3, color='red', linestyle='--', alpha=0.5, label='Random (3-way)')
ax.set_xlabel('K (shots por clase)')
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy vs K-shot\n(clases nunca vistas)', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()
print("✅ Más shots → mejor accuracy, pero el salto mayor es de 1 a 5 shots")

## 4. Siamese Networks

### Idea
Entrenar una red que recibe **pares** de ejemplos y predice si pertenecen a la **misma clase**:

$$p(\text{match}) = \sigma(- \| f_\theta(x_1) - f_\theta(x_2) \|)$$

### Contrastive Loss
$$\mathcal{L} = (1-y) \cdot \frac{1}{2} d^2 + y \cdot \frac{1}{2} \max(0, m - d)^2$$

Donde:
- $y = 0$: mismo clase → minimizar distancia
- $y = 1$: clase diferente → maximizar distancia (hasta margen $m$)
- $d = \|f_\theta(x_1) - f_\theta(x_2)\|$

### En inferencia
Para clasificar un nuevo ejemplo, se compara contra cada ejemplo del support set y se asigna la clase del más cercano (nearest neighbor en el espacio aprendido).

In [ ]:
# ============================================================
# Siamese Network desde cero
# ============================================================

class SiameseNetwork:
    """Siamese Network con Contrastive Loss."""
    
    def __init__(self, input_dim, embedding_dim=32, lr=0.01):
        self.encoder = RedEmbedding(input_dim, hidden_dim=64, embedding_dim=embedding_dim)
        self.lr = lr
        self.losses = []
    
    def _generar_pares(self, X, y, n_pares=100):
        """Genera pares positivos (misma clase) y negativos (clase diferente)."""
        pares_X1, pares_X2, labels = [], [], []
        clases = np.unique(y)
        
        for _ in range(n_pares // 2):
            # Par positivo (misma clase)
            c = np.random.choice(clases)
            idx = np.where(y == c)[0]
            i, j = np.random.choice(idx, 2, replace=False)
            pares_X1.append(X[i]); pares_X2.append(X[j]); labels.append(0)
            
            # Par negativo (clase diferente)
            c1, c2 = np.random.choice(clases, 2, replace=False)
            i = np.random.choice(np.where(y == c1)[0])
            j = np.random.choice(np.where(y == c2)[0])
            pares_X1.append(X[i]); pares_X2.append(X[j]); labels.append(1)
        
        return np.array(pares_X1), np.array(pares_X2), np.array(labels)
    
    def _contrastive_loss(self, emb1, emb2, labels, margin=1.0):
        """Contrastive Loss."""
        d = np.sqrt(np.sum((emb1 - emb2) ** 2, axis=1) + 1e-10)
        loss_pos = (1 - labels) * 0.5 * d ** 2
        loss_neg = labels * 0.5 * np.maximum(0, margin - d) ** 2
        return np.mean(loss_pos + loss_neg)
    
    def entrenar(self, X, y, n_epochs=100, n_pares=200, verbose=True):
        """Entrena la Siamese Network."""
        if verbose:
            print("=== Entrenamiento Siamese Network ===\n")
        
        params = self.encoder.get_params()
        
        for epoch in range(n_epochs):
            X1, X2, labels = self._generar_pares(X, y, n_pares)
            
            # Forward
            emb1 = self.encoder.forward(X1)
            emb2_backup = self.encoder.forward(X2)
            loss = self._contrastive_loss(emb1, emb2_backup, labels)
            self.losses.append(loss)
            
            # Numerical gradient (simplificado)
            eps = 1e-4
            for p_idx, param in enumerate(params):
                grad = np.zeros_like(param)
                n_sample = min(param.size, 150)
                indices = np.random.choice(param.size, n_sample, replace=False)
                
                flat = param.ravel()
                for idx in indices:
                    orig = flat[idx]
                    flat[idx] = orig + eps
                    e1 = self.encoder.forward(X1)
                    e2 = self.encoder.forward(X2)
                    l_plus = self._contrastive_loss(e1, e2, labels)
                    
                    flat[idx] = orig - eps
                    e1 = self.encoder.forward(X1)
                    e2 = self.encoder.forward(X2)
                    l_minus = self._contrastive_loss(e1, e2, labels)
                    
                    flat[idx] = orig
                    grad.ravel()[idx] = (l_plus - l_minus) / (2 * eps)
                
                params[p_idx] = param - self.lr * grad
            
            self.encoder.set_params(params)
            
            if verbose and (epoch + 1) % 25 == 0:
                print(f"  Epoch {epoch+1:3d}: loss={loss:.4f}")
        
        if verbose:
            print(f"\n→ Entrenamiento completado")
    
    def predecir_few_shot(self, support_X, support_y, query_X):
        """Clasificación few-shot: nearest neighbor en espacio de embeddings."""
        emb_support = self.encoder.forward(support_X)
        emb_query = self.encoder.forward(query_X)
        
        # Distancias query → support
        dists = np.sum((emb_query[:, np.newaxis, :] - emb_support[np.newaxis, :, :]) ** 2, axis=2)
        
        # Nearest neighbor
        nn_idx = np.argmin(dists, axis=1)
        return support_y[nn_idx]

# --- Entrenar ---
# Usar solo clases de meta-train
mask_train = np.isin(y_digits, clases_train)
X_train_meta = X_digits[mask_train]
y_train_meta = y_digits[mask_train]

siamese = SiameseNetwork(input_dim=64, embedding_dim=16, lr=0.005)
siamese.entrenar(X_train_meta, y_train_meta, n_epochs=100, n_pares=100)

# Evaluar en clases nuevas
gen_test_s = GeneradorTareasFewShot(X_digits, y_digits, n_way=3, k_shot=5, q_queries=10)
accs_siamese = []
for _ in range(50):
    s_X, s_y, q_X, q_y, _ = gen_test_s.generar_episodio(clases_test)
    preds = siamese.predecir_few_shot(s_X, s_y, q_X)
    accs_siamese.append(np.mean(preds == q_y))

print(f"\n🎯 Siamese Network en clases nuevas: {np.mean(accs_siamese):.4f} ± {np.std(accs_siamese):.4f}")

## 5. MAML — Model-Agnostic Meta-Learning

### La idea más influyente en meta-learning (Finn et al., 2017)

MAML aprende una **inicialización de pesos** $\theta$ que, con **pocos pasos de gradiente** en una tarea nueva, produce un buen modelo.

### Algoritmo

```
Para cada iteración del meta-entrenamiento:
    1. Muestrear batch de tareas {T₁, T₂, ..., Tₙ}
    2. Para cada tarea Tᵢ:
        a. INNER LOOP: adaptar θ al support set con K pasos de gradiente
           θ'ᵢ = θ - α ∇θ L(θ, Sᵢ)           ← gradient descent local
        b. Evaluar θ'ᵢ en el query set Qᵢ
    3. OUTER LOOP: actualizar θ usando los gradientes del query set
       θ ← θ - β ∇θ Σᵢ L(θ'ᵢ, Qᵢ)           ← meta-gradient
```

### Fórmula clave
$$\theta \leftarrow \theta - \beta \nabla_\theta \sum_{\mathcal{T}_i} \mathcal{L}(\theta - \alpha \nabla_\theta \mathcal{L}(\theta, S_i), Q_i)$$

### Intuición geométrica
MAML busca un punto $\theta$ en el espacio de parámetros que está **cerca de la solución óptima** de muchas tareas simultáneamente:

```
                    θ*₁ (óptimo tarea 1)
                   ╱
     θ (MAML) ────
                   ╲
                    θ*₂ (óptimo tarea 2)
```

Un paso de gradiente desde $\theta$ llega a un buen $\theta'$ para cada tarea.

In [ ]:
# ============================================================
# MAML desde cero (Model-Agnostic Meta-Learning)
# ============================================================

class ModeloLineal:
    """Modelo lineal simple para MAML (para claridad didáctica)."""
    
    def __init__(self, input_dim, output_dim):
        self.W = np.random.randn(input_dim, output_dim) * 0.01
        self.b = np.zeros(output_dim)
    
    def forward(self, X):
        logits = X @ self.W + self.b
        # Softmax
        exp_l = np.exp(logits - logits.max(axis=1, keepdims=True))
        self.probs = exp_l / exp_l.sum(axis=1, keepdims=True)
        return self.probs
    
    def loss_y_grad(self, X, y):
        """Calcula cross-entropy loss y gradientes analíticos."""
        probs = self.forward(X)
        n = len(y)
        
        # Cross-entropy loss
        loss = -np.sum(np.log(probs[np.arange(n), y] + 1e-10)) / n
        
        # Gradiente: dL/dlogits = probs - one_hot(y)
        d_logits = probs.copy()
        d_logits[np.arange(n), y] -= 1
        d_logits /= n
        
        # dL/dW = X^T @ d_logits, dL/db = sum(d_logits)
        dW = X.T @ d_logits
        db = d_logits.sum(axis=0)
        
        return loss, (dW, db)
    
    def get_params(self):
        return (self.W.copy(), self.b.copy())
    
    def set_params(self, params):
        self.W, self.b = params[0].copy(), params[1].copy()
    
    def accuracy(self, X, y):
        probs = self.forward(X)
        return np.mean(np.argmax(probs, axis=1) == y)


class MAML:
    """MAML: Model-Agnostic Meta-Learning."""
    
    def __init__(self, input_dim, output_dim, inner_lr=0.1, outer_lr=0.001, 
                 inner_steps=5):
        self.modelo = ModeloLineal(input_dim, output_dim)
        self.inner_lr = inner_lr    # α: learning rate del inner loop
        self.outer_lr = outer_lr    # β: learning rate del outer loop
        self.inner_steps = inner_steps
        self.meta_losses = []
    
    def _inner_loop(self, support_X, support_y):
        """Inner loop: adaptar el modelo a una tarea específica.
        
        Retorna los parámetros adaptados θ'.
        """
        # Clonar parámetros actuales
        W_adapted = self.modelo.W.copy()
        b_adapted = self.modelo.b.copy()
        
        modelo_temp = ModeloLineal(support_X.shape[1], W_adapted.shape[1])
        modelo_temp.set_params((W_adapted, b_adapted))
        
        # K pasos de gradient descent en el support set
        for step in range(self.inner_steps):
            loss, (dW, db) = modelo_temp.loss_y_grad(support_X, support_y)
            W_adapted = modelo_temp.W - self.inner_lr * dW
            b_adapted = modelo_temp.b - self.inner_lr * db
            modelo_temp.set_params((W_adapted, b_adapted))
        
        return modelo_temp
    
    def _outer_step(self, episodios):
        """Outer loop: meta-actualización de θ usando gradientes del query set."""
        meta_dW = np.zeros_like(self.modelo.W)
        meta_db = np.zeros_like(self.modelo.b)
        meta_loss = 0
        
        for s_X, s_y, q_X, q_y, _ in episodios:
            # Inner loop: adaptar al support set
            modelo_adaptado = self._inner_loop(s_X, s_y)
            
            # Evaluar en query set
            loss_query, (dW_query, db_query) = modelo_adaptado.loss_y_grad(q_X, q_y)
            meta_loss += loss_query
            
            # Aproximación de primer orden (FOMAML):
            # Usar directamente el gradiente del query como meta-gradiente
            meta_dW += dW_query
            meta_db += db_query
        
        n_tareas = len(episodios)
        meta_dW /= n_tareas
        meta_db /= n_tareas
        meta_loss /= n_tareas
        
        # Actualizar meta-parámetros
        self.modelo.W -= self.outer_lr * meta_dW
        self.modelo.b -= self.outer_lr * meta_db
        
        return meta_loss
    
    def entrenar(self, generador, n_iteraciones=300, tareas_por_iter=4, 
                 clases_train=None, verbose=True):
        """Meta-entrenamiento de MAML."""
        if verbose:
            print("=== MAML: Meta-Entrenamiento ===")
            print(f"    Inner LR (α): {self.inner_lr}")
            print(f"    Outer LR (β): {self.outer_lr}")
            print(f"    Inner steps:  {self.inner_steps}")
            print(f"    Tareas/iter:  {tareas_por_iter}\n")
        
        for it in range(n_iteraciones):
            episodios = generador.generar_lote(tareas_por_iter, clases_train)
            meta_loss = self._outer_step(episodios)
            self.meta_losses.append(meta_loss)
            
            if verbose and (it + 1) % 50 == 0:
                avg_loss = np.mean(self.meta_losses[-50:])
                # Evaluar accuracy rápido
                s_X, s_y, q_X, q_y, _ = generador.generar_episodio(clases_train)
                modelo_ad = self._inner_loop(s_X, s_y)
                acc = modelo_ad.accuracy(q_X, q_y)
                print(f"  Iter {it+1:4d}: meta-loss={avg_loss:.4f}, acc_episodio={acc:.4f}")
        
        if verbose:
            print(f"\n→ Meta-entrenamiento completado")
    
    def adaptar_y_predecir(self, support_X, support_y, query_X):
        """Adapta a una tarea nueva y predice."""
        modelo_adaptado = self._inner_loop(support_X, support_y)
        probs = modelo_adaptado.forward(query_X)
        return np.argmax(probs, axis=1)
    
    def evaluar(self, generador, n_episodios=100, clases_test=None):
        """Evalúa en tareas con clases nunca vistas."""
        accuracies = []
        for _ in range(n_episodios):
            s_X, s_y, q_X, q_y, _ = generador.generar_episodio(clases_test)
            preds = self.adaptar_y_predecir(s_X, s_y, q_X)
            accuracies.append(np.mean(preds == q_y))
        return np.mean(accuracies), np.std(accuracies)

# --- Entrenar MAML ---
gen_maml = GeneradorTareasFewShot(X_digits, y_digits, n_way=3, k_shot=5, q_queries=10)
maml = MAML(input_dim=64, output_dim=3, inner_lr=0.1, outer_lr=0.005, inner_steps=5)
maml.entrenar(gen_maml, n_iteraciones=300, tareas_por_iter=3, clases_train=clases_train)

# Evaluar
acc_maml, std_maml = maml.evaluar(gen_maml, n_episodios=50, clases_test=clases_test)
print(f"\n🎯 MAML en clases NUEVAS: {acc_maml:.4f} ± {std_maml:.4f}")

In [ ]:
# ============================================================
# Visualización: MAML inner loop adaptation
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Meta-training loss
ax = axes[0]
window = 20
losses_smooth = np.convolve(maml.meta_losses, np.ones(window)/window, mode='valid')
ax.plot(losses_smooth, 'b-', lw=1.5)
ax.set_xlabel('Meta-iteración'); ax.set_ylabel('Meta-Loss')
ax.set_title('MAML: Curva de Meta-Entrenamiento', fontweight='bold')
ax.grid(True, alpha=0.3)

# 2. Efecto del número de inner steps
ax = axes[1]
inner_steps_range = [0, 1, 2, 3, 5, 10]
accs_by_steps = []

for steps in inner_steps_range:
    maml_temp = MAML(64, 3, inner_lr=0.1, outer_lr=0.005, inner_steps=steps)
    # Copiar parámetros meta-entrenados
    maml_temp.modelo.set_params(maml.modelo.get_params())
    
    acc, std = maml_temp.evaluar(gen_maml, n_episodios=30, clases_test=clases_test)
    accs_by_steps.append((acc, std))

accs_s = [a[0] for a in accs_by_steps]
stds_s = [a[1] for a in accs_by_steps]
ax.errorbar(inner_steps_range, accs_s, yerr=stds_s, fmt='ro-', lw=2, capsize=5, markersize=8)
ax.axhline(1/3, color='gray', linestyle='--', alpha=0.5, label='Random')
ax.set_xlabel('Inner Steps (pasos de adaptación)')
ax.set_ylabel('Accuracy')
ax.set_title('MAML: Efecto de Inner Steps\n(más steps = más adaptación)', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)

# 3. Adaptación progresiva en una tarea
ax = axes[2]
s_X, s_y, q_X, q_y, clases_ep = gen_maml.generar_episodio(clases_test)

# Rastrear accuracy en cada step del inner loop
accs_inner = []
modelo_temp = ModeloLineal(64, 3)
modelo_temp.set_params(maml.modelo.get_params())

for step in range(15):
    acc = modelo_temp.accuracy(q_X, q_y)
    accs_inner.append(acc)
    loss, (dW, db) = modelo_temp.loss_y_grad(s_X, s_y)
    modelo_temp.W -= 0.1 * dW
    modelo_temp.b -= 0.1 * db

ax.plot(accs_inner, 'go-', lw=2, markersize=6)
ax.axhline(1/3, color='red', linestyle='--', alpha=0.5, label='Random')
ax.set_xlabel('Inner Step')
ax.set_ylabel('Accuracy en Query Set')
ax.set_title(f'Adaptación a tarea nueva\n(clases {list(clases_ep)})', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()
print("✅ MAML: pocos inner steps ya mejoran significativamente sobre random")
print("   La inicialización meta-aprendida es el factor clave")

## 6. Reptile — MAML Simplificado

### Idea (Nichol et al., 2018)
Reptile es una versión más simple de MAML que **no requiere calcular gradientes de segundo orden**:

### Algoritmo
```
Para cada iteración:
    1. Muestrear tarea Tᵢ
    2. Copiar parámetros: θ' = θ
    3. Entrenar θ' en Tᵢ por K pasos de SGD  →  θ'_K
    4. Actualizar meta-parámetros: θ ← θ + ε(θ'_K - θ)
```

### ¿Por qué funciona?
Reptile mueve $\theta$ hacia las soluciones de tareas individuales. El promedio de estas direcciones converge a un punto que está **cerca de buenas soluciones** para todas las tareas.

### Comparación MAML vs Reptile

| Aspecto | MAML | Reptile |
|---------|------|---------|
| Gradientes | Segundo orden (o FOMAML) | Solo primer orden |
| Implementación | Compleja | Muy simple |
| Memoria | Alta (guardar grafo computacional) | Baja |
| Rendimiento | Ligeramente mejor | Casi igual |
| Aplicabilidad | Cualquier modelo diferenciable | Cualquier modelo diferenciable |

In [ ]:
# ============================================================
# Reptile desde cero
# ============================================================

class Reptile:
    """Reptile: meta-learning simplificado."""
    
    def __init__(self, input_dim, output_dim, inner_lr=0.1, meta_lr=0.1,
                 inner_steps=10):
        self.modelo = ModeloLineal(input_dim, output_dim)
        self.inner_lr = inner_lr
        self.meta_lr = meta_lr
        self.inner_steps = inner_steps
        self.meta_losses = []
    
    def entrenar(self, generador, n_iteraciones=500, clases_train=None, verbose=True):
        """Meta-entrenamiento con Reptile."""
        if verbose:
            print("=== Reptile: Meta-Entrenamiento ===")
            print(f"    Inner LR: {self.inner_lr}, Meta LR: {self.meta_lr}")
            print(f"    Inner steps: {self.inner_steps}\n")
        
        for it in range(n_iteraciones):
            # 1. Guardar parámetros iniciales
            W_init, b_init = self.modelo.get_params()
            
            # 2. Muestrear tarea y adaptar
            s_X, s_y, q_X, q_y, _ = generador.generar_episodio(clases_train)
            
            for step in range(self.inner_steps):
                loss, (dW, db) = self.modelo.loss_y_grad(s_X, s_y)
                self.modelo.W -= self.inner_lr * dW
                self.modelo.b -= self.inner_lr * db
            
            # Evaluar en query
            loss_query, _ = self.modelo.loss_y_grad(q_X, q_y)
            self.meta_losses.append(loss_query)
            
            # 3. Meta-update: θ ← θ + ε(θ' - θ)
            W_adapted, b_adapted = self.modelo.get_params()
            
            # Interpolación hacia la solución de la tarea
            new_W = W_init + self.meta_lr * (W_adapted - W_init)
            new_b = b_init + self.meta_lr * (b_adapted - b_init)
            self.modelo.set_params((new_W, new_b))
            
            if verbose and (it + 1) % 100 == 0:
                avg_loss = np.mean(self.meta_losses[-100:])
                print(f"  Iter {it+1:4d}: meta-loss={avg_loss:.4f}")
        
        if verbose:
            print(f"\n→ Entrenamiento completado")
    
    def adaptar_y_predecir(self, support_X, support_y, query_X, n_steps=10):
        """Adapta y predice para una nueva tarea."""
        # Clonar modelo
        modelo_temp = ModeloLineal(support_X.shape[1], self.modelo.W.shape[1])
        modelo_temp.set_params(self.modelo.get_params())
        
        # Adaptar
        for _ in range(n_steps):
            _, (dW, db) = modelo_temp.loss_y_grad(support_X, support_y)
            modelo_temp.W -= self.inner_lr * dW
            modelo_temp.b -= self.inner_lr * db
        
        probs = modelo_temp.forward(query_X)
        return np.argmax(probs, axis=1)
    
    def evaluar(self, generador, n_episodios=100, clases_test=None, n_steps=10):
        accuracies = []
        for _ in range(n_episodios):
            s_X, s_y, q_X, q_y, _ = generador.generar_episodio(clases_test)
            preds = self.adaptar_y_predecir(s_X, s_y, q_X, n_steps)
            accuracies.append(np.mean(preds == q_y))
        return np.mean(accuracies), np.std(accuracies)

# --- Entrenar Reptile ---
gen_reptile = GeneradorTareasFewShot(X_digits, y_digits, n_way=3, k_shot=5, q_queries=10)
reptile = Reptile(input_dim=64, output_dim=3, inner_lr=0.1, meta_lr=0.1, inner_steps=10)
reptile.entrenar(gen_reptile, n_iteraciones=500, clases_train=clases_train)

# Evaluar
acc_reptile, std_reptile = reptile.evaluar(gen_reptile, n_episodios=50, clases_test=clases_test)
print(f"\n🎯 Reptile en clases NUEVAS: {acc_reptile:.4f} ± {std_reptile:.4f}")

## 7. Meta-Learning para Selección de Algoritmos

### Idea (Auto-sklearn, Rice's Framework)
Usar meta-learning para predecir **qué algoritmo funcionará mejor** en un dataset nuevo, basándose en **meta-features** del dataset.

### Meta-Features
Características que describen un dataset sin mirar los datos individuales:
- **Simples**: n_samples, n_features, n_clases, ratio, % missing
- **Estadísticas**: media/std de correlaciones, skewness, kurtosis
- **Landmarking**: accuracy de modelos simples (1-NN, Decision Stump, etc.)
- **Information-theoretic**: entropía de clases, información mutua promedio

### Proceso
1. **Offline**: Evaluar M modelos en D datasets → tabla (D × M) de scores
2. **Extraer meta-features** de cada dataset → tabla (D × F)
3. **Entrenar meta-modelo**: meta-features → ranking de algoritmos
4. **Nuevo dataset**: extraer meta-features → predecir mejores modelos → warm start

In [ ]:
# ============================================================
# Meta-Learning para Selección de Algoritmos desde cero
# ============================================================

class MetaLearnerAlgoritmos:
    """Aprende a recomendar algoritmos según las características del dataset."""
    
    def __init__(self):
        self.meta_base = []  # (meta_features, scores_por_modelo)
        self.nombres_modelos = []
    
    def extraer_meta_features(self, X, y):
        """Extrae características que describen el dataset."""
        n, p = X.shape
        clases = np.unique(y)
        n_clases = len(clases)
        
        # Estadísticas básicas
        mf = {
            'log_n': np.log(n),
            'log_p': np.log(p),
            'ratio_np': n / p,
            'n_clases': n_clases,
        }
        
        # Balance de clases
        counts = np.bincount(y.astype(int), minlength=n_clases)
        probs = counts / n
        mf['entropia_clases'] = -np.sum(probs * np.log2(probs + 1e-10))
        mf['balance_ratio'] = counts.min() / (counts.max() + 1e-10)
        
        # Estadísticas de features
        stds = X.std(axis=0)
        mf['media_std'] = np.mean(stds)
        mf['cv_std'] = np.std(stds) / (np.mean(stds) + 1e-10)
        
        # Correlaciones
        if p > 1:
            corr = np.abs(np.corrcoef(X.T))
            np.fill_diagonal(corr, 0)
            mf['media_corr'] = np.nanmean(corr)
            mf['max_corr'] = np.nanmax(corr)
        else:
            mf['media_corr'] = 0
            mf['max_corr'] = 0
        
        # Skewness
        media = X.mean(axis=0)
        skew = np.mean(np.abs((X - media) ** 3) / (stds ** 3 + 1e-10), axis=0)
        mf['media_skew'] = np.mean(skew)
        
        # Landmarking: accuracy de modelos simples
        try:
            mf['landmark_1nn'] = cross_val_score(
                KNeighborsClassifier(1), X, y, cv=2, scoring='accuracy').mean()
        except:
            mf['landmark_1nn'] = 0.5
        
        try:
            from sklearn.tree import DecisionTreeClassifier
            mf['landmark_stump'] = cross_val_score(
                DecisionTreeClassifier(max_depth=1), X, y, cv=2, scoring='accuracy').mean()
        except:
            mf['landmark_stump'] = 0.5
        
        return mf
    
    def _mf_to_vector(self, mf):
        keys = sorted(mf.keys())
        return np.array([mf[k] for k in keys])
    
    def construir_base(self, datasets, modelos):
        """Construye la base de meta-conocimiento."""
        self.nombres_modelos = list(modelos.keys())
        
        print("=== Construyendo Base de Meta-Conocimiento ===\n")
        
        for i, (nombre_ds, X, y) in enumerate(datasets):
            # Meta-features
            mf = self.extraer_meta_features(X, y)
            
            # Evaluar cada modelo
            scores = {}
            for nombre_m, modelo in modelos.items():
                try:
                    score = cross_val_score(modelo, X, y, cv=3, scoring='accuracy').mean()
                except:
                    score = 0.0
                scores[nombre_m] = score
            
            self.meta_base.append({
                'nombre': nombre_ds,
                'meta_features': mf,
                'scores': scores
            })
            
            mejor = max(scores, key=scores.get)
            print(f"  Dataset '{nombre_ds}' (n={X.shape[0]}, p={X.shape[1]}): "
                  f"mejor={mejor} ({scores[mejor]:.3f})")
        
        print(f"\n→ Base construida: {len(self.meta_base)} datasets × {len(self.nombres_modelos)} modelos")
    
    def recomendar(self, X_nuevo, y_nuevo, top_n=3):
        """Recomienda modelos para un dataset nuevo."""
        mf_nuevo = self.extraer_meta_features(X_nuevo, y_nuevo)
        vec_nuevo = self._mf_to_vector(mf_nuevo)
        
        # Vectores de la base
        vecs_base = np.array([self._mf_to_vector(e['meta_features']) for e in self.meta_base])
        
        # Normalizar
        all_vecs = np.vstack([vecs_base, vec_nuevo.reshape(1, -1)])
        media = all_vecs.mean(axis=0)
        std = all_vecs.std(axis=0) + 1e-10
        vecs_norm = (all_vecs - media) / std
        
        vec_nuevo_norm = vecs_norm[-1]
        vecs_base_norm = vecs_norm[:-1]
        
        # Distancias
        dists = np.linalg.norm(vecs_base_norm - vec_nuevo_norm, axis=1)
        pesos = 1.0 / (dists + 0.01)
        pesos /= pesos.sum()
        
        # Score ponderado por modelo
        score_por_modelo = {m: 0 for m in self.nombres_modelos}
        for peso, entrada in zip(pesos, self.meta_base):
            for modelo, score in entrada['scores'].items():
                score_por_modelo[modelo] += peso * score
        
        # Ranking
        ranking = sorted(score_por_modelo.items(), key=lambda x: x[1], reverse=True)
        
        # Dataset más similar
        idx_similar = np.argmin(dists)
        
        return ranking[:top_n], self.meta_base[idx_similar]['nombre'], mf_nuevo

# --- Construir base de conocimiento ---
np.random.seed(42)

# Generar datasets variados
datasets_meta = []
configs = [
    ('Pocos features, lineal', 300, 5, 3, 0.05),
    ('Muchos features', 200, 50, 2, 0.1),
    ('Grande y fácil', 1000, 10, 2, 0.01),
    ('Pequeño y difícil', 100, 20, 4, 0.2),
    ('Balanceado', 500, 15, 3, 0.05),
    ('Muy correlacionado', 400, 8, 2, 0.1),
    ('Alta dimensión', 150, 100, 3, 0.15),
    ('Muchas clases', 600, 12, 6, 0.1),
]

for nombre, n, p, c, flip in configs:
    X_gen, y_gen = make_classification(n_samples=n, n_features=p, 
                                        n_informative=max(2, p//3),
                                        n_classes=min(c, max(2, p//3)),
                                        n_clusters_per_class=1,
                                        flip_y=flip, random_state=42)
    X_gen = StandardScaler().fit_transform(X_gen)
    datasets_meta.append((nombre, X_gen, y_gen))

modelos_candidatos = {
    'LogReg': LogisticRegression(max_iter=500),
    'RF': RandomForestClassifier(n_estimators=50, random_state=42),
    'GBM': GradientBoostingClassifier(n_estimators=50, random_state=42),
    'SVM': SVC(),
    'KNN': KNeighborsClassifier(),
}

meta_algo = MetaLearnerAlgoritmos()
meta_algo.construir_base(datasets_meta, modelos_candidatos)

# Recomendar para un dataset nuevo
X_test_ds, y_test_ds = make_classification(n_samples=350, n_features=25, n_informative=10,
                                            n_classes=3, random_state=99)
X_test_ds = StandardScaler().fit_transform(X_test_ds)

print(f"\n=== Recomendación para dataset nuevo ===")
print(f"    (n=350, p=25, 3 clases)")
ranking, ds_similar, mf = meta_algo.recomendar(X_test_ds, y_test_ds, top_n=5)
print(f"\nDataset más similar: '{ds_similar}'")
print(f"\nRanking recomendado:")
for modelo, score in ranking:
    print(f"  {modelo:10s}: score ponderado = {score:.4f}")

# Verificar con evaluación real
print(f"\nVerificación (accuracy real):")
for nombre, modelo in modelos_candidatos.items():
    score_real = cross_val_score(modelo, X_test_ds, y_test_ds, cv=5, scoring='accuracy').mean()
    print(f"  {nombre:10s}: {score_real:.4f}")

## 8. Comparación: Todos los Métodos de Meta-Learning

### Resumen de enfoques implementados

| Método | Tipo | Cómo aprende | Ventaja |
|--------|------|-------------|---------|
| **Prototypical Net** | Metric | Espacio de embeddings con prototipos | Simple, eficaz |
| **Siamese Net** | Metric | Pares + contrastive loss | Flexible |
| **MAML** | Optimization | Inicialización óptima para adaptación rápida | Model-agnostic |
| **Reptile** | Optimization | Interpolación hacia soluciones de tareas | Muy simple |
| **Meta-Algoritmos** | Task-level | Meta-features → ranking de modelos | Práctico, escalable |

### ¿Cuándo usar cada uno?
- **Pocos datos, muchas tareas similares** → Prototypical Networks
- **Necesidad de flexibilidad en el modelo** → MAML
- **Implementación rápida** → Reptile
- **Selección de modelo en producción** → Meta-Learning para algoritmos
- **Similaridad entre pares** → Siamese Networks

In [ ]:
# ============================================================
# Comparación experimental: todos los métodos
# ============================================================

print("=== Comparación en Few-Shot Learning (3-way 5-shot, clases 7,8,9) ===\n")

# Baseline: KNN sin meta-learning
accs_knn = []
gen_comp = GeneradorTareasFewShot(X_digits, y_digits, n_way=3, k_shot=5, q_queries=10)
for _ in range(50):
    s_X, s_y, q_X, q_y, _ = gen_comp.generar_episodio(clases_test)
    knn = KNeighborsClassifier(n_neighbors=1)
    knn.fit(s_X, s_y)
    accs_knn.append(accuracy_score(q_y, knn.predict(q_X)))

# Baseline: LogReg sin meta-learning
accs_lr = []
for _ in range(50):
    s_X, s_y, q_X, q_y, _ = gen_comp.generar_episodio(clases_test)
    lr = LogisticRegression(max_iter=500)
    lr.fit(s_X, s_y)
    accs_lr.append(accuracy_score(q_y, lr.predict(q_X)))

# Recopilar resultados
resultados = {
    'Random': (1/3, 0),
    '1-NN (sin meta)': (np.mean(accs_knn), np.std(accs_knn)),
    'LogReg (sin meta)': (np.mean(accs_lr), np.std(accs_lr)),
    'Siamese Network': (np.mean(accs_siamese), np.std(accs_siamese)),
    'Prototypical Net': (acc_test, std_test),
    'MAML (FOMAML)': (acc_maml, std_maml),
    'Reptile': (acc_reptile, std_reptile),
}

for nombre, (acc, std) in resultados.items():
    bar = '█' * int(acc * 40)
    print(f"  {nombre:20s}: {acc:.4f} ± {std:.4f}  {bar}")

# Visualizar
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Barras comparativas
ax = axes[0]
nombres = list(resultados.keys())
accs = [resultados[n][0] for n in nombres]
stds = [resultados[n][1] for n in nombres]
colores = ['lightgray', 'lightcoral', 'lightsalmon', 'skyblue', 'steelblue', 'royalblue', 'navy']

bars = ax.barh(range(len(nombres)), accs, xerr=stds, color=colores, 
               alpha=0.8, edgecolor='black', capsize=3)
ax.set_yticks(range(len(nombres))); ax.set_yticklabels(nombres, fontsize=10)
ax.set_xlabel('Accuracy'); ax.set_xlim(0, 1.05)
ax.set_title('Comparación de Métodos\n(3-way 5-shot, clases nuevas)', fontweight='bold')
ax.axvline(1/3, color='red', linestyle='--', alpha=0.5, label='Random')
ax.legend(); ax.invert_yaxis()

# 2. Accuracy vs K-shot para cada método
ax = axes[1]
k_shots = [1, 3, 5, 10]
metodos_few = {
    '1-NN': lambda s_X, s_y, q_X: KNeighborsClassifier(1).fit(s_X, s_y).predict(q_X),
    'Prototypical': lambda s_X, s_y, q_X: proto_net.predecir(s_X, s_y, q_X),
    'MAML': lambda s_X, s_y, q_X: maml.adaptar_y_predecir(s_X, s_y, q_X),
    'Reptile': lambda s_X, s_y, q_X: reptile.adaptar_y_predecir(s_X, s_y, q_X),
}

for nombre_m, predictor in metodos_few.items():
    accs_k = []
    for k in k_shots:
        gen_k = GeneradorTareasFewShot(X_digits, y_digits, n_way=3, k_shot=k, q_queries=10)
        accs_ep = []
        for _ in range(30):
            s_X, s_y, q_X, q_y, _ = gen_k.generar_episodio(clases_test)
            try:
                preds = predictor(s_X, s_y, q_X)
                accs_ep.append(np.mean(preds == q_y))
            except:
                pass
        accs_k.append(np.mean(accs_ep) if accs_ep else 1/3)
    ax.plot(k_shots, accs_k, 'o-', lw=2, markersize=7, label=nombre_m)

ax.axhline(1/3, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('K (shots por clase)'); ax.set_ylabel('Accuracy')
ax.set_title('Accuracy vs K-shot\n(3-way, clases nuevas)', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)

# 3. Curvas de entrenamiento
ax = axes[2]
w = 20
if proto_net.losses:
    losses_p = np.convolve(proto_net.losses, np.ones(w)/w, mode='valid')
    ax.plot(losses_p, label='Prototypical', lw=1.5)
if maml.meta_losses:
    losses_m = np.convolve(maml.meta_losses, np.ones(w)/w, mode='valid')
    ax.plot(losses_m, label='MAML', lw=1.5)
if reptile.meta_losses:
    losses_r = np.convolve(reptile.meta_losses, np.ones(w)/w, mode='valid')
    ax.plot(losses_r, label='Reptile', lw=1.5)

ax.set_xlabel('Episodio / Iteración'); ax.set_ylabel('Loss (suavizado)')
ax.set_title('Curvas de Meta-Entrenamiento', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# Experimento: Few-Shot en Wine dataset
# ============================================================

from sklearn.datasets import load_wine

wine = load_wine()
X_wine = StandardScaler().fit_transform(wine.data)
y_wine = wine.target

# Split: clase 0 y 1 para meta-train, clase 2 para meta-test
clases_train_w = np.array([0, 1])
clases_test_w = np.array([2])  # Solo 1 clase → usaremos 1-way (detección)

# Para hacer N-way: crear sub-clases artificiales dividiendo las existentes
# Alternativa: usar todas las clases pero con few-shot cross-validation

print("=== Few-Shot Learning en Wine Dataset ===\n")
print(f"Features: {X_wine.shape[1]}, Clases: {np.unique(y_wine)}")
print(f"Ejemplos por clase: {np.bincount(y_wine)}")

# Evaluar con diferentes K-shots usando MAML
maml_wine = MAML(input_dim=13, output_dim=3, inner_lr=0.05, outer_lr=0.005, inner_steps=5)
gen_wine = GeneradorTareasFewShot(X_wine, y_wine, n_way=3, k_shot=5, q_queries=5)

# Entrenar rápido
maml_wine.entrenar(gen_wine, n_iteraciones=200, verbose=False)

# Comparar: entrenar desde cero vs MAML
print("Comparación: k ejemplos por clase\n")
print(f"{'K':>3s}  {'1-NN':>8s}  {'LogReg':>8s}  {'MAML':>8s}")
print("-" * 35)

for k in [1, 2, 3, 5, 10, 20]:
    accs = {'1nn': [], 'lr': [], 'maml': []}
    
    gen_k = GeneradorTareasFewShot(X_wine, y_wine, n_way=3, k_shot=k, q_queries=5)
    
    for _ in range(30):
        s_X, s_y, q_X, q_y, _ = gen_k.generar_episodio()
        
        # 1-NN
        knn = KNeighborsClassifier(1).fit(s_X, s_y)
        accs['1nn'].append(accuracy_score(q_y, knn.predict(q_X)))
        
        # LogReg
        try:
            lr = LogisticRegression(max_iter=500).fit(s_X, s_y)
            accs['lr'].append(accuracy_score(q_y, lr.predict(q_X)))
        except:
            accs['lr'].append(1/3)
        
        # MAML
        preds = maml_wine.adaptar_y_predecir(s_X, s_y, q_X)
        accs['maml'].append(np.mean(preds == q_y))
    
    print(f"{k:3d}  {np.mean(accs['1nn']):8.4f}  {np.mean(accs['lr']):8.4f}  {np.mean(accs['maml']):8.4f}")

print(f"\n→ MAML se beneficia especialmente con muy pocos datos (K=1,2,3)")

## Resumen: Meta-Learning

### Mapa conceptual

```
                    META-LEARNING
                    "Aprender a aprender"
                          │
         ┌────────────────┼────────────────┐
         │                │                │
    METRIC-BASED    OPTIMIZATION      MODEL-BASED
         │           -BASED                │
    ┌────┴────┐    ┌────┴────┐      ┌────┴────┐
    │Siamese  │    │ MAML    │      │  MANN   │
    │Matching │    │ Reptile │      │  SNAIL  │
    │Proto Net│    │Meta-SGD │      │ MetaNet │
    └─────────┘    └─────────┘      └─────────┘
```

### Lo que implementamos desde cero

| Componente | Clase | Clave |
|------------|-------|-------|
| Generador de tareas | `GeneradorTareasFewShot` | N-way K-shot episodios |
| Red de embeddings | `RedEmbedding` | Forward pass MLP |
| Prototypical Networks | `PrototypicalNetwork` | Prototipos = centroides, distancia euclidiana |
| Siamese Network | `SiameseNetwork` | Pares + contrastive loss |
| MAML (FOMAML) | `MAML` | Inner loop (adaptar) + Outer loop (meta-actualizar) |
| Reptile | `Reptile` | Interpolación θ + ε(θ' - θ) |
| Meta-Learning Algoritmos | `MetaLearnerAlgoritmos` | Meta-features + ranking ponderado |

### Ideas clave
1. **Few-shot learning**: aprender con 1-5 ejemplos por clase
2. **Episodic training**: simular durante entrenamiento la situación de test
3. **Meta-train/meta-test split**: las clases de test NUNCA se ven en entrenamiento
4. **Inner/outer loop**: adaptación local vs aprendizaje global
5. **Meta-features**: describir datasets para transferir conocimiento